# 04 Business Conclusions

Финальный ноутбук собирает рассчитанные таблицы, выделяет лучшие и слабые сегменты и формирует осторожные продуктовые выводы.

## 1. Load calculated results

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cohort import generate_product_hypotheses
from src.visualization import save_dataframe

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

retention = pd.read_csv(
    PROCESSED_PATH / "retention_d7_d30_d90.csv",
    parse_dates=["cohort_month"],
)
channel_retention = pd.read_csv(PROCESSED_PATH / "channel_retention.csv")
cohort_size = pd.read_csv(
    PROCESSED_PATH / "cohort_size_by_month.csv",
    parse_dates=["cohort_month"],
)

display(retention.head())
display(channel_retention.head())

## 2. Summary of retention metrics

In [ ]:
retention_summary = retention[
    ["d7_retention", "d30_retention", "d90_retention"]
].describe().T

display(retention_summary.round(2))

## 3. Best and worst cohorts

In [ ]:
best_cohorts_by_d30 = retention.nlargest(5, "d30_retention")
worst_cohorts_by_d30 = retention.nsmallest(5, "d30_retention")

print("Best cohorts by D30 retention")
display(best_cohorts_by_d30[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

print("Worst cohorts by D30 retention")
display(worst_cohorts_by_d30[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

## 4. Channel comparison

In [ ]:
best_channels_by_d30 = channel_retention.nlargest(5, "d30_retention")
best_channels_by_revenue = channel_retention.nlargest(5, "revenue_per_customer")

print("Best channels by D30 retention")
display(best_channels_by_d30[["acquisition_channel", "customers_count", "d30_retention", "d90_retention"]].round(2))

print("Best channels by revenue per customer")
display(best_channels_by_revenue[["acquisition_channel", "customers_count", "revenue_per_customer", "average_order_value"]].round(2))

## 5. Product hypotheses

In [ ]:
hypotheses = generate_product_hypotheses(retention, channel_retention)

for index, hypothesis in enumerate(hypotheses, start=1):
    print(f"{index}. {hypothesis}")

## 6. Business recommendations

In [ ]:
best_cohort_label = best_cohorts_by_d30.iloc[0]["cohort_month"].strftime("%Y-%m")
worst_cohort_label = worst_cohorts_by_d30.iloc[0]["cohort_month"].strftime("%Y-%m")
best_channel_label = best_channels_by_d30.iloc[0]["acquisition_channel"]

recommendations = f"""### Рекомендации

- Сравнить первую покупку клиентов из когорт {best_cohort_label} и {worst_cohort_label}: различия могут быть связаны с ассортиментом, промо или сезонностью и требуют проверки.
- Использовать D7 retention как ранний индикатор проблем в первом клиентском опыте.
- Для клиентов без повторной покупки в первые 7 дней протестировать CRM-триггеры, персональные рекомендации и напоминания.
- Канал {best_channel_label} выглядит сильным по D30 retention в синтетической сегментации, но для реальной оценки нужны CAC и marketing spend.
"""

display(Markdown(recommendations))

## 7. Limitations

In [ ]:
limitations = """### Ограничения

- `acquisition_channel` создан синтетически и не отражает реальные маркетинговые источники.
- Без CAC, marketing spend и маржинальности нельзя оценивать окупаемость каналов.
- Retention не доказывает причинность: различия могут быть связаны с сезонностью, промо, ассортиментом или составом аудитории.
- Для объяснения причин оттока нужны дополнительные поведенческие и клиентские данные.
"""

display(Markdown(limitations))

## 8. Save final summary

In [ ]:
save_dataframe(best_cohorts_by_d30, PROCESSED_PATH / "best_cohorts_by_d30.csv")
save_dataframe(worst_cohorts_by_d30, PROCESSED_PATH / "worst_cohorts_by_d30.csv")
save_dataframe(best_channels_by_d30, PROCESSED_PATH / "best_channels_by_d30.csv")
save_dataframe(best_channels_by_revenue, PROCESSED_PATH / "best_channels_by_revenue_per_customer.csv")

final_summary = pd.DataFrame(
    {
        "metric": [
            "best_cohort_by_d30",
            "worst_cohort_by_d30",
            "best_channel_by_d30",
            "best_channel_by_revenue_per_customer",
        ],
        "value": [
            best_cohort_label,
            worst_cohort_label,
            best_channel_label,
            best_channels_by_revenue.iloc[0]["acquisition_channel"],
        ],
    }
)

save_dataframe(final_summary, PROCESSED_PATH / "final_summary_tables.csv")
display(final_summary)

print("Saved final summary tables.")